# 🏔️ Module 02: Loss Geometry, Optimization & SOTA Optimizers
### *From Robbins-Monro (1951) & Polyak (1964) to AdamW (2017) and Cosine Annealing*

---

## 📜 Historical Lineage
The quest to efficiently navigate complex mathematical terrains spans nearly seven decades:
- **Herbert Robbins & Sutton Monro (1951)**: *"A Stochastic Approximation Method"* — invented Stochastic Gradient Descent (SGD) under noisy observational data.
- **Boris T. Polyak (1964)**: *"Some methods of speeding up the convergence of iteration methods"* — introduced Heavy-Ball Momentum, modeling a particle with mass rolling down a potential well to dampen oscillations.
- **Geoffrey Hinton et al. (2012)**: RMSprop (Coursera Lecture 6) — normalized step sizes using exponential moving averages of squared gradients.
- **Diederik P. Kingma & Jimmy Ba (2014)**: *"Adam: A Method for Stochastic Optimization"* (ICLR 2015) — combined first-moment momentum and second-moment RMSprop with bias correction.
- **Ilya Loshchilov & Frank Hutter (2017)**: *"Decoupled Weight Decay Regularization"* (AdamW, ICLR 2019) — discovered that L2 regularization in Adam was broken, decoupled weight decay from adaptive moments, and revolutionized modern Transformer optimization.

---

## ⛷️ The Physical Analogy: Skiing Down a Foggy Mountain Ravine
- **The Fog**: Dense alpine fog obscures all visibility beyond your boots. You only feel the local slope of the snow beneath your skis (the local gradient $\nabla L$).
- **Vanilla SGD**: You point your skis in the direction of steepest descent. In an elongated mountain ravine, you repeatedly crash into the steep canyon walls, ping-ponging back and forth while making almost zero forward progress down the gentle valley floor.
- **Polyak Momentum**: You gain heavy physical momentum and inertia ($v$). The left-right crashes cancel out, allowing forward kinetic energy to propel you smoothly along the ravine.
- **RMSprop / Adam**: Your smart skis automatically detect chatter and roughness on each axis, applying braking friction along the steep canyon walls and loosening friction along the flat canyon floor.
- **AdamW**: A decoupled bungee tether gently retracts you toward the origin, preventing weights from ballooning to dangerous magnitudes.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch

print("Environment initialized for Optimization Study.")


## 1. Mathematical Formulation of the Optimizers

### 1.1 Stochastic Gradient Descent (Robbins & Monro 1951)
$$\theta_{t+1} = \theta_t - \eta g_t$$

### 1.2 Polyak Heavy-Ball Momentum (Polyak 1964)
$$v_{t+1} = \beta v_t + g_t$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

### 1.3 RMSprop (Hinton 2012)
$$s_{t+1} = \alpha s_t + (1 - \alpha) g_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_{t+1}} + \epsilon} g_t$$

### 1.4 Adam (Kingma & Ba 2014)
$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) g_t, \quad v_{t+1} = \beta_2 v_t + (1 - \beta_2) g_t^2$$
$$\hat{m}_{t+1} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}, \quad \hat{v}_{t+1} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$$

### 1.5 AdamW (Loshchilov & Hutter 2017)
Decoupled weight decay update:
$$\theta_{t+1} = \theta_t - \eta \lambda \theta_t - \frac{\eta}{\sqrt{\hat{v}_{t+1}} + \epsilon} \hat{m}_{t+1}$$


In [ ]:
class BaseOptimizer:
    def __init__(self, params, lr=1e-2):
        self.params = params
        self.lr = lr
        self.step_count = 0

class ScratchSGD(BaseOptimizer):
    def step(self, grads):
        self.step_count += 1
        for p, g in zip(self.params, grads):
            p -= self.lr * g

class ScratchMomentum(BaseOptimizer):
    def __init__(self, params, lr=1e-2, beta=0.9):
        super().__init__(params, lr)
        self.beta = beta
        self.v = [np.zeros_like(p) for p in params]
    def step(self, grads):
        self.step_count += 1
        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.v[i] = self.beta * self.v[i] + g
            p -= self.lr * self.v[i]

class ScratchAdamW(BaseOptimizer):
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
    def step(self, grads):
        self.step_count += 1
        t = self.step_count
        for i, (p, g) in enumerate(zip(self.params, grads)):
            # Decoupled weight decay
            if self.weight_decay != 0:
                p -= self.lr * self.weight_decay * p
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (g ** 2)
            m_hat = self.m[i] / (1 - self.beta1 ** t)
            v_hat = self.v[i] / (1 - self.beta2 ** t)
            p -= (self.lr / (np.sqrt(v_hat) + self.eps)) * m_hat


## 2. Navigating the Ill-Conditioned Ravine
Anisotropic curvature: $f(x, y) = 0.5 (x^2 + 50 y^2)$.
The condition number $\kappa = 50$. The steep $y$-axis induces wild oscillations in vanilla SGD, while Momentum and Adam glides straight down the ravine floor.


In [ ]:
def ravine_loss(x, y, gamma=50.0):
    return 0.5 * (x**2 + gamma * y**2)

def ravine_grad(x, y, gamma=50.0):
    return np.array([x, gamma * y])

def run_simulation(opt_cls, start_pos=(-4.0, 1.2), steps=60, **kwargs):
    pos = np.array(start_pos, dtype=np.float64)
    opt = opt_cls([pos], **kwargs)
    path = [pos.copy()]
    for _ in range(steps):
        g = ravine_grad(pos[0], pos[1])
        opt.step([g])
        path.append(pos.copy())
    return np.array(path)

path_sgd = run_simulation(ScratchSGD, lr=0.035)
path_mom = run_simulation(ScratchMomentum, lr=0.035, beta=0.85)
path_adamw = run_simulation(ScratchAdamW, lr=0.15)

# Visualizing Trajectories on Contour Map
X, Y = np.meshgrid(np.linspace(-5, 1, 150), np.linspace(-2, 2, 150))
Z = ravine_loss(X, Y)

plt.figure(figsize=(12, 6))
plt.contour(X, Y, Z, levels=np.logspace(-1, 2.5, 25), cmap='cividis', alpha=0.6)
plt.plot(path_sgd[:, 0], path_sgd[:, 1], 'r.-', label='Vanilla SGD (Robbins-Monro 1951)', lw=1.5, ms=4)
plt.plot(path_mom[:, 0], path_mom[:, 1], 'm.-', label='Polyak Momentum (1964)', lw=2, ms=4)
plt.plot(path_adamw[:, 0], path_adamw[:, 1], 'g.-', label='AdamW (2017)', lw=2.5, ms=5)
plt.plot(0, 0, 'r*', ms=12, label='Global Minimum')
plt.title("Optimization Trajectories in Ill-Conditioned Ravine (Condition Number = 50)")
plt.xlabel("Gentle Canyon Floor (x)")
plt.ylabel("Steep Canyon Walls (y)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. The Rosenbrock Banana Valley
$$f(x, y) = (1 - x)^2 + 100(y - x^2)^2$$
Global minimum at $(1, 1)$. Finding the curving valley is easy; navigating its flat floor is famously difficult.


In [ ]:
def rosenbrock(x, y):
    return (1.0 - x)**2 + 100.0 * (y - x**2)**2

def rosenbrock_grad(x, y):
    dx = -2.0 * (1.0 - x) - 400.0 * x * (y - x**2)
    dy = 200.0 * (y - x**2)
    return np.array([dx, dy])

pos = np.array([-1.2, 1.0])
opt = ScratchAdamW([pos], lr=0.02)
path_rosen = [pos.copy()]
for _ in range(300):
    g = rosenbrock_grad(pos[0], pos[1])
    opt.step([g])
    path_rosen.append(pos.copy())
path_rosen = np.array(path_rosen)

XR, YR = np.meshgrid(np.linspace(-1.5, 1.5, 150), np.linspace(-0.5, 1.5, 150))
ZR = rosenbrock(XR, YR)

plt.figure(figsize=(10, 6))
plt.contour(XR, YR, ZR, levels=np.logspace(-1, 3.5, 30), cmap='magma', alpha=0.7)
plt.plot(path_rosen[:, 0], path_rosen[:, 1], 'cyan', lw=2.5, label='AdamW Path')
plt.plot(1, 1, 'r*', ms=14, label='Minimum (1, 1)')
plt.title("AdamW Navigating the Rosenbrock Banana Valley")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Why AdamW? The L2 Regularization vs Decoupled Decay Breakthrough
In classical SGD, adding an $L_2$ penalty $\frac{1}{2}\lambda \|\theta\|^2$ to the loss is algebraically identical to weight decay:
$$\nabla (L + \frac{1}{2}\lambda \|\theta\|^2) = \nabla L + \lambda \theta \implies \theta_{t+1} = \theta_t - \eta (\nabla L + \lambda \theta_t) = \theta_t - \eta \lambda \theta_t - \eta \nabla L$$

However, in **Adam**, the gradient is divided by $\sqrt{v_t} + \epsilon$:
$$\text{Adam L2 Update: } \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t} + \epsilon} (\nabla L + \lambda \theta_t)$$
Notice the flaw: The weight decay term is divided by $\sqrt{v_t}$!
- Parameters with **massive gradients** receive **almost zero weight decay**.
- Parameters with **tiny gradients** receive **excessive weight decay**.

Loshchilov & Hutter (2017) restored true weight decay by pulling it out of the adaptive gradient operator entirely.


In [ ]:
# PyTorch Verification of Adam vs AdamW
w_adam = torch.tensor([10.0], requires_grad=True)
w_adamw = torch.tensor([10.0], requires_grad=True)

opt_adam = torch.optim.Adam([w_adam], lr=0.1, weight_decay=0.1)
opt_adamw = torch.optim.AdamW([w_adamw], lr=0.1, weight_decay=0.1)

# Large synthetic gradient steps
for _ in range(5):
    (100.0 * w_adam).backward(); opt_adam.step(); opt_adam.zero_grad()
    (100.0 * w_adamw).backward(); opt_adamw.step(); opt_adamw.zero_grad()

print(f"Standard Adam (L2 Regularized) Final Weight: {w_adam.item():.4f}")
print(f"AdamW (Decoupled Weight Decay) Final Weight: {w_adamw.item():.4f}")
print(f"Divergence: {abs(w_adam.item() - w_adamw.item()):.4f}")


## 5. Production SOTA: Cosine Annealing with Warmup
Modern Foundation Models (LLaMA, Chinchilla, GPT-4) pair AdamW with a **Linear Warmup + Cosine Annealing** learning rate schedule.


In [ ]:
from torch.optim.lr_scheduler import _LRScheduler

class CosineWarmupScheduler(_LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6, last_epoch=-1):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(0, self.last_epoch)
        if step < self.warmup_steps:
            alpha = float(step) / float(max(1, self.warmup_steps))
            return [self.min_lr + alpha * (b - self.min_lr) for b in self.base_lrs]
        else:
            decay_steps = max(1, self.total_steps - 1 - self.warmup_steps)
            progress = min(1.0, max(0.0, float(step - self.warmup_steps) / float(decay_steps)))
            cosine_mult = 0.5 * (1.0 + math.cos(math.pi * progress))
            return [self.min_lr + cosine_mult * (b - self.min_lr) for b in self.base_lrs]

dummy = torch.nn.Parameter(torch.zeros(1))
opt = torch.optim.SGD([dummy], lr=1e-3)
sched = CosineWarmupScheduler(opt, warmup_steps=20, total_steps=100, min_lr=1e-5)

lrs = []
for _ in range(100):
    lrs.append(sched.get_lr()[0])
    opt.step()
    sched.step()

plt.figure(figsize=(9, 4))
plt.plot(lrs, color='darkorange', lw=2.5)
plt.axvline(20, color='red', ls='--', label='Warmup End (Step 20)')
plt.title("SOTA Learning Rate Schedule: Linear Warmup (20 steps) + Cosine Annealing (80 steps)")
plt.xlabel("Training Step")
plt.ylabel("Learning Rate")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## 🎯 Key Takeaways from Module 02
1. **Curvature is Anisotropic**: Loss surfaces are ravines, not smooth bowls. Gradients point perpendicular to progress; momentum is essential to cancel orthogonal vibrations.
2. **Saddle Points Dominate**: In high dimensions, local minima are rare; saddles and flat plateaus dominate. Adaptive gradient methods (Adam) accelerate escape along flat dimensions.
3. **AdamW is King**: Decoupled weight decay fixes Adam's mathematical flaw with $L_2$ regularization, providing true regularization to modern neural architectures.
4. **Warmup Prevents Divergence**: High learning rates early on destabilize randomly initialized weights. Linear warmup stabilizes training before cosine annealing guides convergence.
